<a href="https://colab.research.google.com/github/hemanthsaigude/CN7030/blob/main/Potfolio_Exercise_Document_Classification_on_NewsGroup_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer, Word2Vec
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import numpy as np

# Start Spark Session
spark = SparkSession.builder.appName("DocumentClassification_Compare").getOrCreate()

# Fetch 20 Newsgroups Data
newsgroups = fetch_20newsgroups(subset='all')

# Convert the dataset to a DataFrame for PySpark processing
data = pd.DataFrame({'text': newsgroups.data, 'category': newsgroups.target})
df = spark.createDataFrame(data)

print(f"Total number of documents: {len(newsgroups.data)}")
print(f"Number of categories: {len(newsgroups.target_names)}")

# Filter 25% of documents to speed up processing
df_sampled = df.sample(withReplacement=False, fraction=0.25, seed=42)
print(f"Total number of documents after sampling: {df_sampled.count()}")

# Split the data into training and testing sets (80% train, 20% test) early on
# so both models are trained and evaluated on the exact same splits.
train_data, test_data = df_sampled.randomSplit([0.8, 0.2], seed=42)

# --- Step 1: Common Preprocessing Stages ---
tokenizer = Tokenizer(inputCol="text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
indexer = StringIndexer(inputCol="category", outputCol="label", handleInvalid="skip")

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# ==========================================
# MODEL 1: TF-IDF Pipeline
# ==========================================
hashingTF = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=1000)
idf = IDF(inputCol="raw_features", outputCol="features_tfidf")
lr_tfidf = LogisticRegression(featuresCol="features_tfidf", labelCol="label")

# Build and train TF-IDF Pipeline
pipeline_tfidf = Pipeline(stages=[tokenizer, remover, indexer, hashingTF, idf, lr_tfidf])
model_tfidf = pipeline_tfidf.fit(train_data)

# Predict and Evaluate TF-IDF
predictions_tfidf = model_tfidf.transform(test_data)
accuracy_tfidf = evaluator.evaluate(predictions_tfidf)

# ==========================================
# MODEL 2: Word2Vec Pipeline
# ==========================================
word2vec = Word2Vec(vectorSize=100, minCount=1, inputCol="filtered_words", outputCol="features_w2v")
lr_w2v = LogisticRegression(featuresCol="features_w2v", labelCol="label")

# Build and train Word2Vec Pipeline
pipeline_w2v = Pipeline(stages=[tokenizer, remover, indexer, word2vec, lr_w2v])
model_w2v = pipeline_w2v.fit(train_data)

# Predict and Evaluate Word2Vec
predictions_w2v = model_w2v.transform(test_data)
accuracy_w2v = evaluator.evaluate(predictions_w2v)

# ==========================================
# RESULTS & COMPARISON
# ==========================================
print("-" * 40)
print(f"TF-IDF Model Accuracy: {accuracy_tfidf:.4f}")
print(f"Word2Vec Model Accuracy: {accuracy_w2v:.4f}")
print("-" * 40)

# Optional: Keep the Term-Document Matrix extraction from original Lab 2 (using the TF-IDF model)
print("\nExtracting Term-Document Matrix (TDM) from TF-IDF features...")
processed_data = model_tfidf.transform(df_sampled)
tdm_rdd = processed_data.select("features_tfidf").rdd.map(lambda row: row[0].toArray())
tdm_array = np.array(tdm_rdd.collect())
tdm_df = pd.DataFrame(tdm_array)

print("Term-Document Matrix (TDM) Head:")
print(tdm_df.head())

In [3]:
!pip install pyspark

import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer, Word2Vec
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import numpy as np

In [4]:
# Start Spark Session
spark = SparkSession.builder.appName("DocumentClassification_Compare").getOrCreate()

# Fetch 20 Newsgroups Data
newsgroups = fetch_20newsgroups(subset='all')

# Convert the dataset to a DataFrame for PySpark processing
data = pd.DataFrame({'text': newsgroups.data, 'category': newsgroups.target})
df = spark.createDataFrame(data)

In [5]:
print(f"Total number of documents: {len(newsgroups.data)}")
print(f"Number of categories: {len(newsgroups.target_names)}")


Total number of documents: 18846
Number of categories: 20


In [6]:
# Filter 25% of documents to speed up processing
df_sampled = df.sample(withReplacement=False, fraction=0.25, seed=42)
print(f"Total number of documents after sampling: {df_sampled.count()}")

# Split the data into training and testing sets (80% train, 20% test) early on
# so both models are trained and evaluated on the exact same splits.
train_data, test_data = df_sampled.randomSplit([0.8, 0.2], seed=42)

Total number of documents after sampling: 4773


In [7]:
# --- Step 1: Common Preprocessing Stages ---
tokenizer = Tokenizer(inputCol="text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
indexer = StringIndexer(inputCol="category", outputCol="label", handleInvalid="skip")

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

In [ ]:
# --- Step 1: Common Preprocessing Stages ---
tokenizer = Tokenizer(inputCol="text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
indexer = StringIndexer(inputCol="category", outputCol="label", handleInvalid="skip")

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

In [9]:
hashingTF = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=1000)
idf = IDF(inputCol="raw_features", outputCol="features_tfidf")
lr_tfidf = LogisticRegression(featuresCol="features_tfidf", labelCol="label")

# Build and train TF-IDF Pipeline
pipeline_tfidf = Pipeline(stages=[tokenizer, remover, indexer, hashingTF, idf, lr_tfidf])
model_tfidf = pipeline_tfidf.fit(train_data)

# Predict and Evaluate TF-IDF
predictions_tfidf = model_tfidf.transform(test_data)
accuracy_tfidf = evaluator.evaluate(predictions_tfidf)

In [11]:
word2vec = Word2Vec(vectorSize=100, minCount=5, inputCol="filtered_words", outputCol="features_w2v")
lr_w2v = LogisticRegression(featuresCol="features_w2v", labelCol="label")

# Build and train Word2Vec Pipeline
pipeline_w2v = Pipeline(stages=[tokenizer, remover, indexer, word2vec, lr_w2v])
model_w2v = pipeline_w2v.fit(train_data)

# Predict and Evaluate Word2Vec
predictions_w2v = model_w2v.transform(test_data)
accuracy_w2v = evaluator.evaluate(predictions_w2v)

In [12]:
# RESULTS & COMPARISON
# ==========================================
print("-" * 40)
print(f"TF-IDF Model Accuracy: {accuracy_tfidf:.4f}")
print(f"Word2Vec Model Accuracy: {accuracy_w2v:.4f}")
print("-" * 40)

# Optional: Keep the Term-Document Matrix extraction from original Lab 2 (using the TF-IDF model)
print("\nExtracting Term-Document Matrix (TDM) from TF-IDF features...")
processed_data = model_tfidf.transform(df_sampled)
tdm_rdd = processed_data.select("features_tfidf").rdd.map(lambda row: row[0].toArray())
tdm_array = np.array(tdm_rdd.collect())
tdm_df = pd.DataFrame(tdm_array)

print("Term-Document Matrix (TDM) Head:")
print(tdm_df.head())

----------------------------------------
TF-IDF Model Accuracy: 0.5330
Word2Vec Model Accuracy: 0.4838
----------------------------------------

Extracting Term-Document Matrix (TDM) from TF-IDF features...
Term-Document Matrix (TDM) Head:
        0    1        2    3         4    5        6    7         8    9    \
0  0.000000  0.0  1.90774  0.0  0.000000  0.0  0.00000  0.0  0.000000  0.0   
1  0.000000  0.0  0.00000  0.0  0.000000  0.0  0.00000  0.0  0.000000  0.0   
2  0.000000  0.0  0.00000  0.0  2.223593  0.0  2.70702  0.0  6.417494  0.0   
3  1.983452  0.0  0.00000  0.0  0.000000  0.0  0.00000  0.0  0.000000  0.0   
4  0.000000  0.0  0.00000  0.0  0.000000  0.0  0.00000  0.0  0.000000  0.0   

   ...       990  991       992  993       994       995       996       997  \
0  ...  2.100533  0.0  0.000000  0.0  4.574994  0.000000  0.000000  0.000000   
1  ...  0.000000  0.0  0.000000  0.0  0.000000  0.000000  0.000000  0.000000   
2  ...  2.100533  0.0  6.410937  0.0  0.000000  0.0